# 🐦 Post to X (Twitter) with an Image from Cloudinary

This notebook demonstrates how to:
1. Fetch an image from a **Cloudinary URL**
2. Upload it to the **X Media API** to get a `media_id`
3. Post a **tweet** with the image attached

---
### Requirements
```
pip install requests
```
You'll need **OAuth 1.0a** credentials from the [X Developer Portal](https://developer.x.com):
- `API_KEY` (Consumer Key)
- `API_SECRET` (Consumer Secret)
- `ACCESS_TOKEN`
- `ACCESS_TOKEN_SECRET`

## 1. Setup — Install & Import

In [ ]:
!pip install requests requests-oauthlib

In [1]:
import os
import requests
import base64
import json
from dotenv import load_dotenv
from requests_oauthlib import OAuth1

load_dotenv()

True

## 2. Credentials — Fill in your X API keys

In [4]:
API_KEY             = os.environ["X_API_KEY"]
API_SECRET          = os.environ["X_API_SECRET"]
ACCESS_TOKEN        = os.environ["X_ACCESS_TOKEN"]
ACCESS_TOKEN_SECRET = os.environ["X_ACCESS_SECRET"]

# OAuth1 auth object (used for media upload — v1.1 endpoint)
auth = OAuth1(API_KEY, API_SECRET, ACCESS_TOKEN, ACCESS_TOKEN_SECRET)

## 3. Step 1 — Fetch image from Cloudinary and encode as base64

In [5]:
# Your Cloudinary image URL
CLOUDINARY_IMAGE_URL = "https://res.cloudinary.com/dczz0ag3m/image/upload/v1777515344/scatter_plots/scatter_Defensive_Contributions_vs_Final_Third_Passing_2025_2026.png"

def fetch_image_as_base64(url: str) -> tuple[str, str]:
    """
    Fetches an image from a URL and returns:
      - base64 encoded string
      - media_type (e.g. 'image/png')
    """
    response = requests.get(url)
    response.raise_for_status()
    
    media_type = response.headers.get("Content-Type", "image/png").split(";")[0]
    image_b64  = base64.b64encode(response.content).decode("utf-8")
    
    print(f"✅ Image fetched — type: {media_type}, size: {len(response.content) / 1024:.1f} KB")
    return image_b64, media_type

image_b64, media_type = fetch_image_as_base64(CLOUDINARY_IMAGE_URL)

✅ Image fetched — type: image/png, size: 188.3 KB


## 4. Step 2 — Upload the image to X Media API (v2)

POST to `https://api.x.com/2/media/upload` with the base64 image.
Returns a `media_id` to attach to the tweet.

In [6]:
MEDIA_UPLOAD_URL = "https://api.x.com/2/media/upload"

def upload_media_to_x(image_b64: str, media_type: str, auth: OAuth1) -> str:
    """
    Uploads a base64-encoded image to X Media API v2.
    Returns the media_id string.
    """
    payload = {
        "media": image_b64,
        "media_category": "tweet_image",
        "media_type": media_type,
        "shared": False
    }

    response = requests.post(
        MEDIA_UPLOAD_URL,
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        auth=auth
    )

    if response.status_code != 200:
        print(f"❌ Media upload failed: {response.status_code}")
        print(response.text)
        response.raise_for_status()

    data     = response.json()
    media_id = data["data"]["id"]
    expires  = data["data"].get("expires_after_secs", "N/A")

    print(f"✅ Media uploaded successfully!")
    print(f"   media_id        : {media_id}")
    print(f"   expires_after   : {expires}s")
    return media_id

media_id = upload_media_to_x(image_b64, media_type, auth)

✅ Media uploaded successfully!
   media_id        : 2049958759124930561
   expires_after   : 86400s


## 5. Step 3 — Post the tweet with the image attached

In [7]:
TWEET_URL = "https://api.x.com/2/tweets"

def post_tweet_with_image(text: str, media_id: str, auth: OAuth1) -> dict:
    """
    Posts a tweet with an attached image using the X API v2.
    """
    payload = {
        "text": text,
        "media": {
            "media_ids": [media_id]
        }
    }

    response = requests.post(
        TWEET_URL,
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        auth=auth
    )

    if response.status_code not in (200, 201):
        print(f"❌ Tweet failed: {response.status_code}")
        print(response.text)
        response.raise_for_status()

    data = response.json()
    tweet_id = data["data"]["id"]

    print(f"✅ Tweet posted successfully!")
    print(f"   Tweet ID : {tweet_id}")
    print(f"   URL      : https://x.com/i/web/status/{tweet_id}")
    return data

# ✏️ Edit your tweet text here
TWEET_TEXT = """
⚽ All leagues midfielders for Defensive Contributions vs Final Third Passing (2025/2026)
""".strip()

result = post_tweet_with_image(TWEET_TEXT, media_id, auth)

✅ Tweet posted successfully!
   Tweet ID : 2049958976503111841
   URL      : https://x.com/i/web/status/2049958976503111841


## 6. Full Pipeline — One function to rule them all

Convenience wrapper: pass a Cloudinary URL + tweet text, get a live tweet.

In [ ]:
def post_cloudinary_image_to_x(cloudinary_url: str, tweet_text: str, auth: OAuth1) -> dict:
    """
    Full pipeline:
      1. Fetch image from Cloudinary URL
      2. Upload to X Media API
      3. Post tweet with image attached
    
    Returns the full tweet response.
    """
    print("📥 Step 1/3 — Fetching image from Cloudinary...")
    image_b64, media_type = fetch_image_as_base64(cloudinary_url)

    print("\n📤 Step 2/3 — Uploading image to X...")
    media_id = upload_media_to_x(image_b64, media_type, auth)

    print("\n🐦 Step 3/3 — Posting tweet...")
    return post_tweet_with_image(tweet_text, media_id, auth)


# --- Example usage ---
result = post_cloudinary_image_to_x(
    cloudinary_url = "https://res.cloudinary.com/dczz0ag3m/image/upload/v1777564542/bar_charts/bar_Goals_from_Throw-ins_by_Team.png",
    tweet_text     = "⚽ Goals from throw-ins in the PL 2025/26 — only 10 teams have managed it! #PremierLeague",
    auth           = auth
)

---
## Notes

| Step | Endpoint | Auth |
|------|----------|------|
| Upload media | `POST https://api.x.com/2/media/upload` | OAuth 1.0a |
| Post tweet | `POST https://api.x.com/2/tweets` | OAuth 1.0a |

- The `media_id` **expires after ~86400 seconds** — upload and tweet in the same session.
- Supported image types: `image/jpeg`, `image/png`, `image/webp`, `image/gif`
- Max image size for `tweet_image`: **5 MB**
- You can attach **up to 4 images** per tweet by passing multiple `media_ids` in the array.